# AAGI Central Africa 2026 — reproduction notebook

This notebook reproduces the processed data, key statistics, and figures for the
AAGI 2026 Central Africa analysis, using **only** the two supplied source files
under `data/raw/`.

**Read `docs/methodology_and_limitations.md` first.** Two limits govern everything:

1. Comparative scores come from the report's **page-2 summary** (composites are
   *approximate*; pillar scores are one-decimal).
2. The supplied workbook has only **8 populated indicator rows of 80**, so it
   audits *completeness only*.

All cross-country statistics use **n = 6** and are **descriptive, not inferential**.

## 1. Setup

In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROC = ROOT / "data" / "processed"

if not (PROC / "report_summary_with_tiers.csv").exists():
    import subprocess, sys
    subprocess.run([sys.executable, str(ROOT / "code" / "aagi_analysis.py")], check=True)

summary = pd.read_csv(PROC / "report_summary_with_tiers.csv")
summary

## 2. Composite ranking and tiers

In [ ]:
ranking = summary[["Country", "Composite", "Tier"]].sort_values("Composite", ascending=False)
print("Regional mean composite:", round(summary["Composite"].mean(), 3))
print("All below the Established threshold of 3.0:", bool((summary["Composite"] < 3.0).all()))
ranking

## 3. Regional pillar means — the implementation gap

In [ ]:
pillars = ["P1 Strategy","P2 Governance","P3 Infrastructure","P4 Human Capital",
           "P5 Innovation","P6 Ethics","P7 Regional","P8 Implementation"]
means = summary[pillars].mean().sort_values(ascending=False).round(2)
print("Strongest pillar:", means.index[0], "=", means.iloc[0])
print("Weakest pillar:  ", means.index[-1], "=", means.iloc[-1])
means

## 4. Strategy-to-implementation gap by country

In [ ]:
gap = summary[["Country","P1 Strategy","P8 Implementation"]].copy()
gap["Gap"] = (gap["P1 Strategy"] - gap["P8 Implementation"]).round(2)
gap.sort_values("Gap", ascending=False)

## 5. Composite reproducibility check

Reported composites reproduce as the unweighted 8-pillar mean within ±0.05 for
every country **except Gabon**. See `code/verify_composites.py`.

In [ ]:
df = pd.read_csv(PROC / "report_comparative_summary.csv")
df["mean8"] = df[pillars].mean(axis=1).round(3)
df["reported"] = df["Composite"]
df["diff"] = (df["mean8"] - df["reported"]).round(3)
df[["Country","mean8","reported","diff"]]

## 6. Figures

Rendered charts are in `figures/png/` and `figures/svg/`.

In [ ]:
from IPython.display import Image, display
for name in ["01_composite_ranking", "02_pillar_heatmap", "03_mean_pillars"]:
    p = ROOT / "figures" / "png" / f"{name}.png"
    if p.exists():
        display(Image(filename=str(p)))

## 7. Workbook completeness (audit only)

The workbook is a partial template: only P1 and P2 carry four populated rows each.

In [ ]:
pd.read_csv(PROC / "workbook_pillar_completeness.csv")